# 80. MoE Expert Parallel Benchmark | MoE 专家并行基准
**难度：** Hard | **环境：** CPU-first | **标签：** `并行通信`, `MoE`, `基准对比` | **目标人群：** 项目决策练习者

---

## 本节导读

本节要求你验证专家并行带来的吞吐收益是否足以覆盖通信和负载不均衡成本。固定 expert 数量、路由规则和 workload 后，比较 baseline 与 expert parallel 的吞吐、all-to-all 通信量、expert 负载分布和训练稳定性。最终给出是否采用该并行方案的 benchmark 结论。
**主责与复用边界：** 本项目主责是 MoE token dispatch、expert 负载和 all-to-all 通信；显存优化只关注 expert 参数分摊与 activation / 通信缓冲的容量影响，推理优化只复用路由和吞吐观察，不在本项目内替代 71 的 MLA 或 70 的 serving 调度验证。

**关键词：** `MoE`, `expert parallel`, `communication`, `imbalance`, `delivery`

---

## 前置阅读

**导语：** 先把 MoE 路由、负载均衡、专家并行机制和基础并行 benchmark 理顺，再进入这个项目；本节默认你已经知道 expert routing 的基本对象，重点转向专家并行是否值得保留。
- [06. MoE Router | MoE 路由](./06_MoE_Router.ipynb)
- [07. MoE Load Balancing Loss | MoE 负载均衡损失](./07_MoE_Load_Balancing_Loss.ipynb)
- [47. MoE Expert Parallel | MoE 专家并行](./47_MoE_Expert_Parallel.ipynb)
- [79. Distributed Parallel Benchmark | 分布式并行基准](./79_Distributed_Parallel_Benchmark.ipynb)

---

### Step 1：建立 Expert Parallel 的对照问题

- 固定 expert 数、router 策略、batch size、seq len 和训练步数。
- 明确 baseline 是无 expert parallel 还是其他并行策略。
- 统一记录通信时间、负载不均、吞吐和训练稳定性。

### Step 2：固定路由与通信条件

- 专家并行 benchmark 不能脱离 baseline 并行口径单独比较。
- 如果 baseline 的通信统计或吞吐波动本身不稳定，候选结果就没有解释空间。
- 至少要先确认 baseline 的通信时间、吞吐和负载分布是可复现的。

### Step 3：比较吞吐、通信与负载

- 专家并行项目必须同时看吞吐、通信代价、负载不均和训练稳定性，不能只挑单项吞吐收益下结论。
- 但副作用会体现在 all-to-all 通信、负载不均和训练噪声上。
- 如果吞吐变好，但负载不均明显恶化，候选通常只能进入 `tune`，而不是直接 `accept`。

### Step 4：CPU 实现与选型决策

- 专家并行最终不是输出“吞吐有没有涨”，而是输出这套 expert parallel 配置在当前预算下是否值得继续保留、微调或放弃。
- 最终结论建议统一为 `accept / tune / reject`。
- 若进入 `tune`，下一轮优先回调 router、capacity factor、expert 分组或通信拓扑，而不是盲目继续加 expert。

#### 图解：06-07-46-47-79 如何收束到 80 MoE 基准

`80` 把 MoE 的路由、均衡、通信和并行基线收成一个项目页。

```text
06 Router         expert routing / token dispatch
      │
07 Load balance   auxiliary balancing pressure
      │
46 NCCL           communication hotspot evidence
      │
47 Expert parallel dispatch / gather and communication cost
      │
79 Parallel bench baseline distributed throughput
      │
      ▼
80 MoE bench      throughput + imbalance + communication + delivery decision
```

项目页最小产物：

| 模块 | 必须记录 | 用途 |
|:---|:---|:---|
| baseline | 吞吐、通信时间、负载分布 | 保证比较合法 |
| candidate | expert 配置、通信变化、负载变化 | 解释收益来源 |
| 对比 | throughput gain、comm delta、imbalance delta | 判断是否值得 adopt |
| 决策 | accept / tune / reject | 输出 benchmark 结论 |


In [ ]:
from typing import Dict, List


In [ ]:
# 3 个核心 TODO：workload 汇总、baseline 对比、项目判断
# 目标：把通信与负载均衡结果整理成 benchmark 报告

def summarize_moe_parallel_runs(runs: list[dict[str, float]]) -> dict[str, object]:
    """null"""
    # total_tokens = ???；mean_imbalance = ???；mean_comm_ms = ???；stable_runs = ???。
    raise NotImplementedError("请先完成 TODO 代码！")

def compare_moe_parallel_to_baseline(baseline: dict[str, float], candidate: dict[str, float]) -> dict[str, float]:
    """null"""
    # throughput_delta = ???；memory_delta = ???；communication_delta = ???；imbalance_delta = ???。
    raise NotImplementedError("请先完成 TODO 代码！")

def recommend_moe_parallel_run(
    baseline: dict[str, float],
    candidate: dict[str, float],
    max_imbalance: float,
    min_stability: float,
) -> dict[str, object]:
    """null"""
    # imbalance_ok = ???；stability_ok = ???；decision = ???；reason = ???；next_action = ???。
    raise NotImplementedError("请先完成 TODO 代码！")


In [ ]:
# 测试你的实现
def test_moe_parallel_benchmark_template():
    baseline = {'name': 'baseline', 'comm_ms': 40, 'imbalance': 0.25, 'throughput': 100, 'stability': 0.82}
    candidate = {'name': 'expert_parallel', 'comm_ms': 48, 'imbalance': 0.12, 'throughput': 128, 'stability': 0.80}
    summary = summarize_moe_parallel_runs([baseline, candidate])
    assert summary['run_count'] == 2
    assert summary['best_throughput_run'] == 'expert_parallel'
    comparison = compare_moe_parallel_to_baseline(baseline, candidate)
    assert comparison['comm_delta_ms'] == 8
    assert comparison['imbalance_delta'] == -0.13
    assert comparison['throughput_gain'] == 28
    assert comparison['stability_delta'] == -0.02
    decision = recommend_moe_parallel_run(baseline, candidate, max_imbalance=0.15, min_stability=0.78)
    assert decision['decision'] == 'accept'
    assert decision['next_action'] == 'promote_to_cluster_eval'


test_moe_parallel_benchmark_template()
print('测试通过：MoE 专家并行基准模板可以工作。')


---

🛑 **STOP HERE** 🛑
<br><br><br><br><br><br><br><br><br><br>
> 请先尝试自己完成代码并跑通测试。<br>
> 如果你正在 Colab 中运行，并且遇到困难没有思路，可以向下滚动查看参考答案。
<br><br><br><br><br><br><br><br><br><br>

---


## 参考代码与解析

### 代码


In [ ]:
def summarize_moe_parallel_runs(runs: list[dict[str, float]]) -> dict[str, object]:
    best = max(runs, key=lambda item: item.get('throughput', 0.0))
    avg_imbalance = sum(item.get('imbalance', 0.0) for item in runs) / len(runs) if runs else 0.0
    return {'run_count': len(runs), 'best_throughput_run': best.get('name', 'run'), 'avg_imbalance': avg_imbalance}


def compare_moe_parallel_to_baseline(baseline: dict[str, float], candidate: dict[str, float]) -> dict[str, float]:
    return {
        'comm_delta_ms': candidate.get('comm_ms', 0.0) - baseline.get('comm_ms', 0.0),
        'imbalance_delta': round(candidate.get('imbalance', 0.0) - baseline.get('imbalance', 0.0), 4),
        'throughput_gain': candidate.get('throughput', 0.0) - baseline.get('throughput', 0.0),
        'stability_delta': round(candidate.get('stability', 0.0) - baseline.get('stability', 0.0), 4),
    }


def recommend_moe_parallel_run(
    baseline: dict[str, float],
    candidate: dict[str, float],
    max_imbalance: float,
    min_stability: float,
) -> dict[str, object]:
    comparison = compare_moe_parallel_to_baseline(baseline, candidate)
    if (
        comparison['throughput_gain'] > 0
        and candidate.get('imbalance', 10**9) <= max_imbalance
        and candidate.get('stability', -10**9) >= min_stability
    ):
        return {'decision': 'accept', 'reason': '吞吐收益、负载均衡和稳定性都达标', 'next_action': 'promote_to_cluster_eval'}
    if comparison['throughput_gain'] > 0 and candidate.get('stability', -10**9) >= min_stability:
        return {'decision': 'tune', 'reason': '吞吐收益可用，但负载不均或通信代价仍偏高', 'next_action': 'refine_router_or_capacity'}
    return {'decision': 'reject', 'reason': '收益不足或训练稳定性不达标', 'next_action': 'fallback_to_parallel_baseline'}


### Step 5（可选）：GPU / 多卡 MoE benchmark

#### 5.1 环境、输入与固定条件

固定 expert 数、top-k、router、capacity factor、batch、sequence length、dtype、GPU 数和互联拓扑。G0 是 dense 或无 expert-parallel baseline，G1 只改变 expert placement / parallel strategy，G2 才比较另一种路由或拓扑。

#### 5.2 环境启动检查

确认 CUDA、PyTorch Distributed、NCCL、GPU 数量和 all-to-all 路径可用。没有多卡或 backend 不支持时保留 failure 记录，不把 CPU 模拟结果写成真实通信证据。

#### 5.3 配置实验条件

为每组记录 expert 数、top-k、capacity factor、world size、router、workload、warmup、repeats、模型版本和结果 JSON。一次只改变并行或路由变量。

#### 5.4 执行实验并保存 JSON

使用固定 token workload 运行 baseline 与 expert-parallel candidate，记录 throughput、all-to-all 时间、dispatch bytes、负载不均衡、peak memory 和稳定性；OOM、通信超时、expert overflow 和启动失败必须保存。

#### 5.5 读取结果与记录证据

| 实验组 | expert / top-k | world size / topology | workload / JSON | throughput | all-to-all | imbalance | peak memory | stability | evidence level | failure | decision |
|---|---|---|---|---:|---:|---:|---:|---|---|---|---|
| G0 baseline | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 | none / 待记录 | pending |
| G1 candidate | 与 G0 对齐 | 与 G0 对齐 | 与 G0 相同 | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 | none / 待记录 | pending |
| G2 optional | 明确差异 | 明确差异 | 与 G0 相同 | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 | 待填写 | none / 待记录 | pending |

#### 5.6 解释结果与形成决策

吞吐提升只有在 all-to-all、负载不均和稳定性仍在预算内时才可 `accept`；通信或 overflow 偏高进入 `tune`，无法启动或收益不成立则为 `reject`。

In [ ]:
# Step 5 GPU 配置：默认关闭；all-to-all 预检与真实 MoE benchmark 分开记录。
RUN_NCCL_SMOKE = False
RUN_REAL_BENCHMARK = False
WORLD_SIZE = 2
EXPERTS = 8
TOP_K = 2
CAPACITY_FACTOR = 1.25
MODEL_ID = 'Qwen/Qwen1.5-MoE-A2.7B'
BATCH_SIZE = 1
SEQ_LEN = 512
WARMUP = 2
REPEATS = 5
SMOKE_PATH = 'benchmarks/results/80_moe_expert_parallel_smoke.json'
RESULT_PATH = 'benchmarks/results/80_moe_expert_parallel_benchmark.json'
# 例如：['torchrun', '--nproc_per_node=2', 'train_moe.py', '--num-experts', '8']
BENCHMARK_COMMAND = None
print({'smoke': RUN_NCCL_SMOKE, 'benchmark': RUN_REAL_BENCHMARK, 'world_size': WORLD_SIZE, 'experts': EXPERTS, 'result': RESULT_PATH})


In [ ]:
# 5.2--5.4：预检 all-to-all；真实 MoE 训练/推理由外部命令写入 RESULT_PATH。
import subprocess
from pathlib import Path

if RUN_NCCL_SMOKE:
    smoke_command = ['torchrun', '--standalone', '--nproc_per_node', str(WORLD_SIZE), 'tools/run_distributed_smoke.py', '--project', '80', '--output', SMOKE_PATH]
    subprocess.run(smoke_command, check=True)
if RUN_REAL_BENCHMARK:
    if not BENCHMARK_COMMAND:
        raise ValueError('请先填写 BENCHMARK_COMMAND；all-to-all 预检不能替代真实 expert dispatch benchmark。')
    subprocess.run(BENCHMARK_COMMAND, check=True)
    if not Path(RESULT_PATH).exists():
        raise FileNotFoundError(f'真实 benchmark 未写入结果文件：{RESULT_PATH}')
if not RUN_NCCL_SMOKE and not RUN_REAL_BENCHMARK:
    print('GPU 实验默认关闭；先确认 all-to-all 路径，再运行固定 expert workload。')


In [ ]:
# 5.5：真实 MoE 结果必须同时报告路由负载、通信代价与稳定性。
import json
from pathlib import Path

if Path(RESULT_PATH).exists():
    result = json.loads(Path(RESULT_PATH).read_text())
    required = {'workload', 'experts', 'routing', 'hardware', 'metrics', 'evidence_level', 'failure', 'decision'}
    missing = required - set(result)
    if missing:
        raise ValueError(f'结果 JSON 缺少项目证据字段：{sorted(missing)}')
    print(result)
else:
    print(f'尚无真实 benchmark 结果：{RESULT_PATH}')


### 解析

这页现在按 `measure -> compare -> decide` 的最小 MoE 专家并行项目闭环组织，不再只是单独比较吞吐和通信代价。

#### TODO 1

- 实现方式：先汇总 run 数量和平均负载不均衡，再找出吞吐最高的 run。
- 关键点：`best_throughput_run` 只是帮助定位最值得回看的 candidate，不等于最终项目结论。
- 项目意义：先把 workload 摘要做平，后面才能在同一 expert 并行设置下比较收益与代价。

#### TODO 2

- 实现方式：统一计算 `comm_delta_ms`、`imbalance_delta`、`throughput_gain` 和 `stability_delta`。
- 关键点：吞吐和稳定性越高越好，通信代价和负载不均衡越低越好，所以指标方向必须统一。
- 项目意义：这一步把 MoE 专家并行从“技巧演示”转成“吞吐、通信和稳定性能否一起成立”的 benchmark 对比。

#### TODO 3

- 实现方式：先复用 baseline 对比结果，再按吞吐收益、imbalance 边界和稳定性输出 `accept / tune / reject`。
- 关键点：`tune` 主要对应吞吐收益已出现，但 router、capacity factor 或通信拓扑还没有一起收稳。
- 项目意义：MoE 专家并行项目最后要回答的是“这套并行方案值不值得继续扩到真实集群”，而不是只看某个吞吐数字。

## 相关阅读

以下资料按“MoE 路由机制 → 专家并行实现 → 通信证据”排列，用于把本节的负载均衡、all-to-all 和吞吐权衡连接到真实系统。

- [Switch Transformers 论文：稀疏专家路由](https://arxiv.org/abs/2101.03961)
- [DeepSpeed-MoE 论文：Scaling Inference with MoE](https://arxiv.org/abs/2201.05596)
- [MegaBlocks 官方仓库](https://github.com/databricks/megablocks)
- [DeepSpeed 官方仓库](https://github.com/microsoft/DeepSpeed)
- [79 分布式并行基准](./79_Distributed_Parallel_Benchmark.ipynb)
- [81 分布式推理项目](./81_Distributed_Inference_Project.ipynb)
- [74 Profiling 驱动的端到端优化](./74_Profiling_Driven_End_to_End_Optimization.ipynb)
